In [ ]:
%pip install -U -r requirements.txt

In [ ]:
from pipeline import Pipeline

pipeline = Pipeline(lookback_periods=30, forward_periods=7)


In [ ]:
reload = False

timeframe = "1d"
candles_file_name = f"ohlcv{timeframe}"
candles_path = f"{pipeline.data_dir}/{candles_file_name}.csv"

if reload:
    candles_df = await pipeline.get_candles_df(timeframe=timeframe)  # noqa: PLE1142
    pipeline.save_csv(candles_file_name, candles_df)

candles_df = pipeline.spark.read.csv(candles_path, header=True, inferSchema=True)

candles_df.show()

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import window as W

# Get oldest and latest timestamps
oldest = candles_df.select(F.min("timestamp")).first()[0]
latest = candles_df.select(F.max("timestamp")).first()[0]
pipeline.logger.info("Data range: %s to %s", oldest, latest)

symbol_window = W.Window.partitionBy("symbol").orderBy("timestamp")
window_size = 17 * 24
rolling_window = symbol_window.rowsBetween(-window_size + 1, 0)


def with_forward_return(df: DataFrame) -> DataFrame:
    forward_window = 7
    forward_window = symbol_window.rowsBetween(1, forward_window)
    return df.withColumn(
        "forward_return", F.exp(F.sum("return").over(forward_window)) - 1
    ).withColumn(
        "price_zscore_fw_return_corr",
        F.corr(F.col("price_zscore"), F.col("forward_return")).over(rolling_window),
    )

return_df = (
    candles_df
    .transform(pipeline.with_returns)
    .transform(pipeline.with_bollinger)
    .transform(pipeline.with_zscore)
    # .transform(pipeline.with_auto_regression)
    # .transform(with_forward_return)
    .transform(pipeline.with_volatility)
    .transform(pipeline.with_beta)
)

return_df.describe().show()
return_df.printSchema()